In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
KEY = os.environ.get("RIOT_API_KEY") 

if KEY is None:
    print("Error: API_KEY environment variable not set.")

In [2]:
import pandas as pd
import itertools
import csv

d_puuids = []
filename = "puuids.csv"

with open(filename, 'r', newline='', encoding='utf-8') as csvfile:
    csv_reader = csv.reader(csvfile, delimiter=',')
    for row in csv_reader:
        d_puuids.append(row)

puuids = list(itertools.chain.from_iterable(d_puuids))


In [3]:
import requests

def get_match_history_ids(puuid, api_key, region, start=0, count=2):
    """Fetches a list of match IDs for a given PUUID."""
    api_url = f"https://{region}.api.riotgames.com/lol/match/v5/matches/by-puuid/{puuid}/ids?start={start}&count={count}&api_key={api_key}"
    response = requests.get(api_url)
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error fetching match IDs: {response.status_code} - {response.text}")
        return None
    
def get_match_details(match_id, api_key, region):
    """Fetches detailed information for a specific match ID."""
    api_url = f"https://{region}.api.riotgames.com/lol/match/v5/matches/{match_id}?api_key={api_key}"
    response = requests.get(api_url)
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error fetching match details for {match_id}: {response.status_code} - {response.text}")
        return None

In [4]:
match_ids = get_match_history_ids(puuids[0], KEY, "americas", 0, 20)

In [5]:
for match in match_ids:
    print(get_match_details(match, KEY, "americas"))

{'metadata': {'dataVersion': '2', 'matchId': 'NA1_5476611031', 'participants': ['J8sJe6LfniGYBt54B4njFFzo_7uO0fugyDW7E_GsFjNNp7qTHffiQGtqvqUsL2a4cRrC1f73v7i9ZQ', 'eDbG-8WYo-oShEN1W1JKratSTOwsk3OJTXDOKdmdClF8GvpQMAJDK1SteqUBJhX6k0-cGSjrZ7IoIw', 'v3RbTWMSyuG31SCjhNTG5hTzrZU39h11lUsJYUbY6Rr8c_phIYdklO5jcNXXhR7h6wkYKPrLiR8diw', 'AIk_iHtODoj62tKIPO_1aUcVGrzdjdMt7uNJDanQNcnOTyi0APS7uXtN_Tzg46t5WePD-wsTQOjKhw', 'TriRfBMsbokrfBz4sqAtx-r3vtNqDCT-KSkmDhqfs5JEqAIi46xZvEnw1z7J870HRDUS3KYnvip0Qg', 'XlVgV6OmDLqcVLApFXi7Hz7oUJt0Ns-kwUs83EeV7Pq3M268qkJqZdI9g1KKW9RdD4923fsirWoYxw', 'fzuGgSU1PqKh6IyLbtE0QvenVzVZnWAvJRq3LRzqJIrHyAZceNsmyF-zEjqvwhcnHR_6_yAriZPGdg', 'MuAywJFIet88TdVRPFO1SAqfoSUYWzmHBp8LGW1wWrDSrAqQ1kQtIOIQSteM3YZLsmVNdk3c4dHy6g', 'WjRZyl-HAf9-bCV74B7u-wAUYrQyKSPeL9eHPqxxnLsZRyoae1Q-k7tqaTJ7BFs9MBaRla3yYlr9gA', 'P2DpONNeaorbnZzfi9Zvi42mO_RMIEi27iC0eLXrOb4dm3n9v-gGeCmtAHsRzcfqaTTtduo1J-lDcA']}, 'info': {'endOfGameResult': 'GameComplete', 'gameCreation': 1769548563484, 'gameDuration': 110, 'g

In [6]:
data = get_match_details(match_ids[3], KEY, "americas")

In [7]:
player_stats = next(p for p in data["info"]["participants"] if p["puuid"] == puuids[0])
len(player_stats)

146

In [8]:
features = [
    "championName", "championId", "teamId",
    "kills", "deaths", "assists", "kda",
    "totalDamageDealtToChampions", "totalDamageTaken",
    "goldEarned", "goldSpent",
    "totalMinionsKilled", "neutralMinionsKilled",
    "visionScore", "wardsPlaced", "wardsKilled", "detectorWardsPlaced",
    "dragonKills", "baronKills", "turretKills", "inhibitorKills",
    "firstBloodKill", "firstTowerKill",
    "totalHeal", "totalHealsOnTeammates",
    "damageSelfMitigated",
    "totalTimeCCDealt", "timeCCingOthers",
    "item0", "item1", "item2", "item3", "item4", "item5", "item6",
    "summoner1Id", "summoner2Id",
    "win", "gameEndedInEarlySurrender", "gameEndedInSurrender",
    "timePlayed",
    "spell1Casts", "spell2Casts", "spell3Casts", "spell4Casts",
]

def get_features(puuid):
    stats = {field: puuid.get(field) for field in features}
    return stats



In [9]:
from datetime import datetime

timestamp = data["info"]["gameStartTimestamp"]
game_start = datetime.fromtimestamp(timestamp / 1000)

player_feats = get_features(player_stats)
player_feats['game_start'] = game_start

player_feats

{'championName': 'Urgot',
 'championId': 6,
 'teamId': 100,
 'kills': 0,
 'deaths': 0,
 'assists': 0,
 'kda': None,
 'totalDamageDealtToChampions': 777,
 'totalDamageTaken': 569,
 'goldEarned': 589,
 'goldSpent': 500,
 'totalMinionsKilled': 2,
 'neutralMinionsKilled': 0,
 'visionScore': 0,
 'wardsPlaced': 0,
 'wardsKilled': 0,
 'detectorWardsPlaced': 0,
 'dragonKills': 0,
 'baronKills': 0,
 'turretKills': 0,
 'inhibitorKills': 0,
 'firstBloodKill': False,
 'firstTowerKill': False,
 'totalHeal': 23,
 'totalHealsOnTeammates': 0,
 'damageSelfMitigated': 337,
 'totalTimeCCDealt': 4,
 'timeCCingOthers': 1,
 'item0': 1055,
 'item1': 0,
 'item2': 0,
 'item3': 0,
 'item4': 0,
 'item5': 0,
 'item6': 3340,
 'summoner1Id': 4,
 'summoner2Id': 14,
 'win': False,
 'gameEndedInEarlySurrender': True,
 'gameEndedInSurrender': False,
 'timePlayed': 94,
 'spell1Casts': 0,
 'spell2Casts': 0,
 'spell3Casts': 2,
 'spell4Casts': 0,
 'game_start': datetime.datetime(2026, 1, 23, 18, 53, 14, 191000)}

In [10]:
import time
time_series = {}
count = 1
for puuid in puuids:
    print(puuid)
    match_ids = get_match_history_ids(puuid, KEY, "americas", 0, 20)
    player_feats = {}
    if match_ids is not None:
        all_matches = []
        for match in match_ids:
            details = get_match_details(match, KEY, "americas")
            if details is not None:
                try:
                    player_stats = next(p for p in details["info"]["participants"] if p["puuid"] == puuid)
                except:
                    print("failed retrieving player_stats")
                    break
                timestamp = details["info"]["gameStartTimestamp"]
                game_start = datetime.fromtimestamp(timestamp / 1000)

                player_feats = get_features(player_stats)
                player_feats['game_start'] = game_start
                all_matches.append(player_feats)
        time_series[puuid] = all_matches
        if count % 3 == 0:
            time.sleep(120)
        count+=1

J8sJe6LfniGYBt54B4njFFzo_7uO0fugyDW7E_GsFjNNp7qTHffiQGtqvqUsL2a4cRrC1f73v7i9ZQ
nsmK2BNffcp--OGQAUv_Is8e4asuQ49p42arV903545Jirg6HleFkb5zckeebbGrdmNaZnbaYLqHQQ
nY-tW8VSjBLVve3OjmtPy24TzWTH6WYc7JiTWWrFD52E-2FPv0NugVi5D1kaAXPbwdS6XQgpLzYxww
Wa_HfFq5JGOPil3nIo248sl-_0flZ35X2-FQEGKrRSXYINaxQytGvQFVXts0J3BGjlr3s9YelzuC8g
XXFNh_5jF6FHij_MYVCu9ryXPnx-0HcOTWz29PwBcrFoJ4ggpUIHetidsVLZGrRcXizShVRzrbPZ_w
CkxHRDzgqIg9a9FAIFkSTVxIngCJ1lzU0KS5Nod6smVanR27nfuljDxIrr2gbGo35I1uF4Ib9_2ijA
tSSh3XdoyJnDS6jmn8iuw2-BANIDkR7MgfTwRAeIv8qLWKmg2BE3eZxNr3AEpKionuXjN5UprDP6jw
4niT6sMU4q2pVZavEoSk1o7hlGts8_z6_UgmOWtxxDgeDOlDYZWCDZ8aOmLFh1fUnVmxPQUH3KdJFQ
failed retrieving player_stats
RsfQgr26nebxq7eMfVEiYWpxI8EkaBNlhz8Mjk_9dH8CwU16Z9DJqb9viAMw-8gQ9PmVkuuCesJd0A
SMjB7m4FJkA2sIwsw4R6qbdaainoThbDDCfulg5JwhzOul7YEG_-Rhm1bbg6uiRsheTCZsJfom8xMQ
JEl8FYsWi2wclR4mCA38JdCbCjQU5H37kSscA_ypkvptdVYdcpvEFtHLwdasnnuI_HG2868lT1x5FA
aOlACZZD7a6Nej9fDuX57KorJMcErXPn84wTSTbj-nllGyWilRkewykMw2Qpa8lelLxll9c8FVmvAA
0ncyG9TY8AuSAXoVuI-2d

In [ ]:
for puuid in time_series:
    for x in time_series[puuid]:
        del x["kda"]
        del x['championName']

In [20]:
import csv


with open("match_data.csv", "w", newline="") as f:
    writer = csv.writer(f)

    for key, values in time_series.items():
        writer.writerow([key] + values)